# Fine-tune the Pedagogical Retrieval Model on GPU

Runs **stage 04 (training) only**. All preprocessing happens locally; all evaluation happens locally afterwards.

**Workflow:**
1. Locally: `python scripts/01_prepare_dataset.py && python scripts/03_create_triplets.py`
2. Here: run all cells, upload `outputs/triplets/triplets_package.zip` when prompted
3. Locally: unzip the downloaded model into `outputs/models/finetuned_pedagogical/`, then `python scripts/05_evaluate.py --compare`

> Runtime > Change runtime type > **GPU** before running.

In [ ]:
# --- Configuration (the only cell you may need to edit) ---
REPO_URL = "https://github.com/YOUR_USERNAME/distractor.git"  # <-- your repository
EPOCHS = 3        # matches src/config.py FINETUNE_EPOCHS
BATCH_SIZE = 32   # matches src/config.py FINETUNE_BATCH_SIZE

In [ ]:
# --- Verify GPU availability ---
import torch
!nvidia-smi -L
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > GPU"
print("GPU OK:", torch.cuda.get_device_name(0))

In [ ]:
# --- Clone repository and install dependencies ---
import os
if not os.path.exists("distractor"):
    !git clone {REPO_URL} distractor
%cd distractor
%pip install -q -r requirements_gpu.txt

In [ ]:
# --- Upload prepared triplets (outputs/triplets/triplets_package.zip from stage 03) ---
import os, zipfile
from google.colab import files

os.makedirs("outputs/triplets", exist_ok=True)
if not os.path.exists("outputs/triplets/train_triplets.jsonl"):
    uploaded = files.upload()  # select triplets_package.zip
    for name in uploaded:
        with zipfile.ZipFile(name) as zf:
            zf.extractall("outputs/triplets")
print(os.listdir("outputs/triplets"))

In [ ]:
# --- Train (stage 04 only) ---
!python scripts/train_gpu.py --epochs {EPOCHS} --batch-size {BATCH_SIZE}

In [ ]:
# --- Zip and download the trained model ---
!cd outputs/models && zip -qr finetuned_pedagogical.zip finetuned_pedagogical -x "*/checkpoints/*"
from google.colab import files
files.download("outputs/models/finetuned_pedagogical.zip")

**Back on your machine:** unzip into `outputs/models/`, then run
```bash
python scripts/05_evaluate.py --compare
```